# Hot Jupiter Sector Inspection

For each Hot Jupiter candidate (5 selected + 5 alternatives), shows **all available TESS sectors**.
Each panel: faint = full LC, bright = longest continuous gap-free segment (gap threshold = 10 min).
Use this to visually pick the best sector per planet.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import lightkurve as lk
import pickle, os, time, warnings
warnings.filterwarnings('ignore')

# ── Constants ─────────────────────────────────────────────────────────────────
RJUP_RSUN  = 0.10045
RSUN_AU    = 0.00465047
GAP_MIN    = 10                  # minutes — gap threshold for continuity
GAP_DAYS   = GAP_MIN / 60 / 24
COLOR_LC   = '#e57373'           # faint full-LC (light red)
COLOR_SEG  = '#c0392b'           # bright best segment (dark red)
CACHE_FILE = '../data/hj_sector_lc_cache.pkl'
OUT_DIR    = '../results'
os.makedirs(OUT_DIR, exist_ok=True)
print('Setup done.')

Setup done.


c:\Users\pnayg\Desktop\cvif-astro-p1\.venv\Lib\site-packages\lightkurve\prf\__init__.py:7: UserWarning: Warning: the tpfmodel submodule is not available without oktopus installed, which requires a current version of autograd. See #1452 for details.
  warnings.warn(


In [2]:
# ── Load planet data ──────────────────────────────────────────────────────────
selected = pd.read_csv('../selected_10_planets.csv')
coverage = pd.read_csv('../data/tess_coverage_raw.csv')
tepcat   = pd.read_csv('../data/planets_ready_for_modeling.csv')

# Recompute depth/T14 with correct unit conversion
for col in ['R_b','R_A','a(AU)','Period']:
    tepcat[col] = pd.to_numeric(tepcat[col], errors='coerce')
tepcat['k']        = tepcat['R_b'] * RJUP_RSUN / tepcat['R_A']
tepcat['aRs']      = tepcat['a(AU)'] / (tepcat['R_A'] * RSUN_AU)
tepcat['depth_pct']= (tepcat['k']**2) * 100
arg = ((1 + tepcat['k']) / tepcat['aRs']).clip(-1, 1)
tepcat['T14_hr']   = (tepcat['Period'] * 24 / np.pi) * np.arcsin(arg)

# --- Build planet list ---
# 5 selected Hot Jupiters
hj_selected = selected[selected['type'] == 'Hot Jupiter'][['System','host_star']].copy()
hj_selected['label'] = 'selected'

# 5 alternative HJ suggestions
hj_alt_names = ['TOI-1408', 'Qatar-1', 'WASP-183', 'WASP-119', 'WASP-062']
hj_alt = tepcat[tepcat['System'].isin(hj_alt_names)][['System','host_star']].drop_duplicates('System').copy()
hj_alt['label'] = 'alternative'

all_hj = pd.concat([hj_selected, hj_alt], ignore_index=True)

# Merge with coverage to get sector lists
cov_sub = coverage[['System','sector_list','n_sectors']]
all_hj  = all_hj.merge(cov_sub, on='System', how='left')

# Merge tepcat params
params_cols = ['System','R_b','Period','depth_pct','T14_hr','Teff']
all_hj = all_hj.merge(tepcat[params_cols].drop_duplicates('System'), on='System', how='left')

print('Hot Jupiter planets to inspect:')
display_cols = ['System','label','R_b','Period','depth_pct','T14_hr','n_sectors']
print(all_hj[display_cols].to_string(index=False))

Hot Jupiter planets to inspect:
    System       label   R_b  Period  depth_pct   T14_hr  n_sectors
  WASP-177    selected 1.580   3.072   3.216085 2.885950          4
     WTS-2    selected 1.363   1.019   3.314789 1.749484          8
Kepler-447    selected 1.650   7.794   2.589365 4.309324         10
   KELT-23    selected 1.322   2.255   1.781216 2.747062         19
    TrES-3    selected 1.310   1.306   2.553379 1.959608          7
  WASP-062 alternative 1.390   4.412   1.274061 3.813762         38
  WASP-119 alternative 1.400   2.500   1.373389 3.296546         28
   Qatar-1 alternative 1.143   1.420   2.044378 1.996770         20
  WASP-183 alternative 1.470   4.112   2.874076 3.218346          4


In [3]:
def parse_sector_list(s):
    """Parse '[21, 44, 46]' string -> list of ints."""
    if pd.isna(s) or str(s).strip() in ('', '[]'):
        return []
    s = str(s).strip().strip('[]')
    return [int(x.strip()) for x in s.split(',') if x.strip().isdigit()]


def lc_to_arrays(lc):
    return {'time': np.array(lc.time.value), 'flux': np.array(lc.flux.value)}


def download_lc_normalized(host, sector, retries=3):
    for attempt in range(retries):
        try:
            sr = lk.search_lightcurve(host, mission='TESS', sector=sector,
                                       author='SPOC', exptime=120)
            if sr is None or len(sr) == 0:
                return None
            lc = sr[0].download(flux_column='pdcsap_flux')
            if lc is None:
                return None
            lc = lc.remove_nans()
            med = np.nanmedian(lc.flux.value)
            return lc_to_arrays(lc) if med == 0 else \
                   {'time': np.array(lc.time.value),
                    'flux': np.array(lc.flux.value) / med}
        except Exception as e:
            if attempt < retries - 1:
                time.sleep(10)
            else:
                print(f'      FAILED S{sector}: {e}')
    return None


def longest_continuous_segment(tarr, farr, gap_days):
    diffs = np.diff(tarr)
    gap_idx = np.where(diffs > gap_days)[0]
    boundaries = np.concatenate([[-1], gap_idx, [len(tarr) - 1]])
    best_len, best_start, best_end = 0, 0, 0
    for i in range(len(boundaries) - 1):
        s = boundaries[i] + 1
        e = boundaries[i + 1]
        seg_len = tarr[e] - tarr[s]
        if seg_len > best_len:
            best_len = seg_len
            best_start = s
            best_end = e
    mask = np.zeros(len(tarr), bool)
    mask[best_start: best_end + 1] = True
    return mask, tarr[best_start], tarr[best_end], best_len


print('Helper functions defined.')

Helper functions defined.


In [ ]:
# ── Load or build cache ───────────────────────────────────────────────────────
lc_cache = {}  # {(planet, sector): {'time':..., 'flux':...}}

if os.path.exists(CACHE_FILE):
    try:
        with open(CACHE_FILE, 'rb') as f:
            lc_cache = pickle.load(f)
        print(f'Loaded cache: {len(lc_cache)} entries')
    except (EOFError, Exception) as e:
        print(f'Cache corrupted ({e}) — starting fresh.')
        os.remove(CACHE_FILE)
        lc_cache = {}

# Download missing sectors
for _, row in all_hj.iterrows():
    planet  = row['System']
    host    = row['host_star']
    sectors = parse_sector_list(row['sector_list'])
    if not sectors:
        print(f'{planet}: no sectors listed')
        continue
    for sec in sectors:
        key = (planet, sec)
        if key in lc_cache:
            continue
        print(f'  Downloading {planet} S{sec}...', end=' ', flush=True)
        data = download_lc_normalized(host, sec)
        if data is not None:
            lc_cache[key] = data
            print(f'{len(data["time"])} pts')
        else:
            lc_cache[key] = None
            print('NO DATA')
        # Atomic save after each planet-sector
        tmp = CACHE_FILE + '.tmp'
        with open(tmp, 'wb') as f:
            pickle.dump(lc_cache, f)
        os.replace(tmp, CACHE_FILE)
        time.sleep(0.3)

print(f'\nCache total: {len(lc_cache)} entries')

12939 pts

12522 pts

In [ ]:
# ── Plot: one block per planet, one panel per sector ─────────────────────────
NCOLS = 5  # sectors per row

for _, row in all_hj.iterrows():
    planet  = row['System']
    sectors = parse_sector_list(row['sector_list'])
    rb      = float(row['R_b'])      if pd.notna(row['R_b'])      else float('nan')
    period  = float(row['Period'])   if pd.notna(row['Period'])   else float('nan')
    depth   = float(row['depth_pct'])if pd.notna(row['depth_pct'])else float('nan')
    t14     = float(row['T14_hr'])   if pd.notna(row['T14_hr'])   else float('nan')
    label   = row['label']

    avail = [s for s in sectors if lc_cache.get((planet, s)) is not None]
    if not avail:
        print(f'{planet}: no data to plot')
        continue

    nrows = int(np.ceil(len(avail) / NCOLS))
    fig, axes = plt.subplots(nrows, NCOLS,
                              figsize=(4.5 * NCOLS, 3.2 * nrows),
                              squeeze=False)
    tag = '★ SELECTED' if label == 'selected' else 'alternative'
    fig.suptitle(
        f"{planet}  [{tag}]  |  R_b={rb:.2f} Rjup  |  P={period:.3f} d  "
        f"|  depth≈{depth:.3f}%  |  T14≈{t14:.2f} hr  |  {len(avail)} sectors",
        fontsize=12, fontweight='bold'
    )

    for idx, sec in enumerate(avail):
        ax  = axes[idx // NCOLS][idx % NCOLS]
        data = lc_cache[(planet, sec)]
        tarr = data['time']
        farr = data['flux']

        seg_mask, seg_t0, seg_t1, seg_days = \
            longest_continuous_segment(tarr, farr, GAP_DAYS)
        seg_pts   = int(seg_mask.sum())
        n_transits = int(seg_days / period) if period > 0 else 0

        # faint full LC
        ax.plot(tarr, farr, '.', color=COLOR_LC, ms=1.0, alpha=0.25, rasterized=True)
        # bright best segment
        ax.plot(tarr[seg_mask], farr[seg_mask], '.', color=COLOR_SEG,
                ms=1.2, alpha=0.8, rasterized=True)
        # shaded span
        ax.axvspan(seg_t0, seg_t1, alpha=0.07, color=COLOR_SEG)

        ax.set_title(
            f'S{sec}  |  {seg_days:.1f}d  |  {seg_pts} pts  |  ~{n_transits} transits',
            fontsize=8
        )
        ax.set_xlabel('BTJD', fontsize=7)
        ax.tick_params(labelsize=7)
        ax.set_ylabel('')

    # Hide unused panels
    for idx in range(len(avail), nrows * NCOLS):
        axes[idx // NCOLS][idx % NCOLS].set_visible(False)

    plt.tight_layout(rect=[0, 0, 1, 0.95])
    fname = planet.replace(' ', '_').replace('/', '_')
    out   = f'{OUT_DIR}/hj_sectors_{fname}.png'
    plt.savefig(out, dpi=120, bbox_inches='tight')
    plt.show()
    print(f'Saved: {out}')